In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 01 — Bronze Layer: Raw Ingestion (v2 — Schema-Fixed)
# MAGIC
# MAGIC **Pipeline**: Unity Catalog Volume CSV → `bronze_facilities_raw` Delta Table
# MAGIC
# MAGIC **Schema**: 46 columns (41 source + 5 metadata). All column names lowercase as per bronze schema.
# MAGIC The ONLY rename allowed is `mongo DB` → `mongo_db` — however the sanitisation function
# MAGIC also lowercases camelCase headers (e.g. `officialWebsite` → `officialwebsite`) which
# MAGIC matches the bronze schema exactly. Silver layer will restore camelCase aliases.

# COMMAND ----------
# MAGIC %md ## 0. Imports & Setup

# COMMAND ----------

import re
from datetime import datetime, timezone
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StringType

spark = SparkSession.builder.getOrCreate()
print(f"Spark version : {spark.version}")
print(f"Run timestamp : {datetime.now(timezone.utc).isoformat()}")

# COMMAND ----------
# MAGIC %md ## 1. Configuration

# COMMAND ----------

class BronzeConfig:
    # Primary path (Unity Catalog Volume)
    VOLUME_PATH   = "/Volumes/virtue_foundation/ghana/volume"
    SOURCE_CSV    = f"{VOLUME_PATH}/Virtue Foundation Ghana v0.3 - Sheet1.csv"
    # FileStore fallback (for users who uploaded via DBFS)
    FILESTORE_CSV = "/FileStore/virtue_foundation/ghana_v0_3.csv"

    CATALOG          = "virtue_foundation"
    SCHEMA           = "ghana"
    BRONZE_TABLE     = "virtue_foundation.ghana.bronze_facilities_raw"

    COUNTRY_SCOPE    = "GH"
    DATASET_VERSION  = "v0.3"
    SOURCE_FILE_NAME = "Virtue_Foundation_Ghana_v0_3_-_Sheet1.csv"

    # Expected schema: 41 CSV cols + 5 metadata = 46 total
    EXPECTED_COLS = 46

    # Exact bronze schema column names (all lowercase)
    BRONZE_COLUMNS = [
        "source_url", "name", "pk_unique_id", "mongo_db", "specialties",
        "procedure", "equipment", "capability", "organization_type",
        "content_table_id", "phone_numbers", "email", "websites",
        "officialwebsite", "yearestablished", "acceptsvolunteers",
        "facebooklink", "twitterlink", "linkedinlink", "instagramlink",
        "logo", "address_line1", "address_line2", "address_line3",
        "address_city", "address_stateorregion", "address_ziporpostcode",
        "address_country", "address_countrycode", "countries",
        "missionstatement", "missionstatementlink", "organizationdescription",
        "facilitytypeid", "operatortypeid", "affiliationtypeids",
        "description", "area", "numberdoctors", "capacity", "unique_id",
        # metadata
        "ingested_at", "source_file", "dataset_version", "country_scope", "row_hash",
    ]

    CSV_OPTIONS = {
        "header":                  "true",
        "multiLine":               "true",
        "escape":                  '"',
        "quote":                   '"',
        "inferSchema":             "false",   # ALL STRING in Bronze
        "encoding":                "UTF-8",
        "ignoreLeadingWhiteSpace": "true",
        "ignoreTrailingWhiteSpace":"true",
        "mode":                    "PERMISSIVE",
    }

cfg = BronzeConfig()
print(f"Primary CSV  : {cfg.SOURCE_CSV}")
print(f"Fallback CSV : {cfg.FILESTORE_CSV}")
print(f"Target table : {cfg.BRONZE_TABLE}")

# COMMAND ----------
# MAGIC %md ## 2. Ensure Catalog & Schema Exist

# COMMAND ----------

spark.sql(f"CREATE CATALOG IF NOT EXISTS {cfg.CATALOG}")
spark.sql(f"USE CATALOG {cfg.CATALOG}")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {cfg.SCHEMA}")
print(f"✅  {cfg.CATALOG}.{cfg.SCHEMA} ready")

# Create Volume if not exists (Unity Catalog)
try:
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {cfg.CATALOG}.{cfg.SCHEMA}.raw")
    print(f"✅  Volume {cfg.CATALOG}.{cfg.SCHEMA}.raw ready")
except Exception as e:
    print(f"Volume creation skipped: {e}")

# COMMAND ----------
# MAGIC %md ## 3. Resolve CSV Path (Volume → FileStore fallback)

# COMMAND ----------

import os

def resolve_csv_path(primary: str, fallback: str) -> str:
    """Try primary path first; fall back to FileStore path."""
    for path in [primary, fallback]:
        try:
            # dbutils.fs.ls works in Databricks; use try/except for local testing
            dbutils.fs.ls(path)
            print(f"✅  Found CSV at: {path}")
            return path
        except Exception:
            print(f"    Not found at: {path}")
    raise FileNotFoundError(
        f"CSV not found at either:\n  {primary}\n  {fallback}\n"
        "Upload the CSV to one of these locations."
    )

CSV_PATH = resolve_csv_path(cfg.SOURCE_CSV, cfg.FILESTORE_CSV)

# COMMAND ----------
# MAGIC %md ## 4. Read Raw CSV (all columns as STRING)

# COMMAND ----------

raw_df = spark.read.options(**cfg.CSV_OPTIONS).csv(CSV_PATH)
print(f"Raw rows    : {raw_df.count():,}")
print(f"Raw columns : {len(raw_df.columns)}")
print("Raw column names:")
for i, c in enumerate(raw_df.columns, 1):
    print(f"  {i:>2}. {c}")

# COMMAND ----------
# MAGIC %md ## 5. Column Name Sanitisation
# MAGIC
# MAGIC Converts all CSV headers to lowercase snake_case.
# MAGIC Key transformation: `mongo DB` → `mongo_db`.
# MAGIC Other camelCase headers (officialWebsite → officialwebsite, etc.) are also lowercased.
# MAGIC This matches the bronze_facilities_raw schema exactly.
# MAGIC The Silver layer will restore specific columns to camelCase.

# COMMAND ----------

def sanitise_col_name(name: str) -> str:
    """Convert any CSV header to safe lowercase snake_case."""
    name = name.strip()
    # Replace spaces and special chars with underscore
    name = re.sub(r"[^a-zA-Z0-9_]", "_", name)
    # Collapse multiple underscores
    name = re.sub(r"_+", "_", name)
    return name.lower().strip("_")

rename_map = {
    col: sanitise_col_name(col)
    for col in raw_df.columns
    if sanitise_col_name(col) != col
}

print(f"Columns renamed ({len(rename_map)}):")
for old, new in rename_map.items():
    print(f"  '{old}'  →  '{new}'")

for old, new in rename_map.items():
    raw_df = raw_df.withColumnRenamed(old, new)

print(f"\nColumns after sanitisation: {len(raw_df.columns)}")

# COMMAND ----------
# MAGIC %md ## 6. Add Bronze Metadata Columns

# COMMAND ----------

# row_hash: MD5 of unique_id || name || description (deterministic, idempotent)
row_hash_expr = F.md5(
    F.concat_ws(
        "||",
        F.coalesce(F.col("unique_id"),   F.lit("")),
        F.coalesce(F.col("name"),        F.lit("")),
        F.coalesce(F.col("description"), F.lit("")),
    )
)

bronze_df = (
    raw_df
    .withColumn("ingested_at",     F.current_timestamp())
    .withColumn("source_file",     F.lit(cfg.SOURCE_FILE_NAME))
    .withColumn("dataset_version", F.lit(cfg.DATASET_VERSION))
    .withColumn("country_scope",   F.lit(cfg.COUNTRY_SCOPE))
    .withColumn("row_hash",        row_hash_expr)
)

print(f"Bronze columns (incl. metadata) : {len(bronze_df.columns)}")

# COMMAND ----------
# MAGIC %md ## 7. Column Validation (exact schema check)

# COMMAND ----------

actual_cols   = set(bronze_df.columns)
expected_cols = set(cfg.BRONZE_COLUMNS)

missing  = expected_cols - actual_cols
extra    = actual_cols   - expected_cols

if missing:
    raise ValueError(f"MISSING bronze columns: {sorted(missing)}")
if extra:
    print(f"⚠️  Extra columns (will be dropped): {sorted(extra)}")

# Reorder to exact schema order
bronze_df = bronze_df.select(*cfg.BRONZE_COLUMNS)

assert len(bronze_df.columns) == cfg.EXPECTED_COLS, (
    f"Column count mismatch: expected {cfg.EXPECTED_COLS}, got {len(bronze_df.columns)}"
)
print(f"✅  Column count validated: {len(bronze_df.columns)}")
print("COLUMN INVENTORY:")
for i, c in enumerate(bronze_df.columns, 1):
    print(f"  {i:>2}. {c}")

# COMMAND ----------
# MAGIC %md ## 8. Row Count & Null Audit

# COMMAND ----------

total_rows = bronze_df.count()
print(f"\n{'='*60}")
print(f"BRONZE QUALITY GATE")
print(f"{'='*60}")
print(f"Total rows     : {total_rows:,}  (expected ~978)")

# Null audit for key columns
KEY_AUDIT_COLS = [
    "name", "unique_id", "specialties", "procedure", "equipment",
    "capability", "facilitytypeid", "address_city", "address_stateorregion",
    "description", "organization_type",
]
print("\nNull counts for key columns:")
for col in KEY_AUDIT_COLS:
    if col in bronze_df.columns:
        null_ct = bronze_df.filter(
            F.col(col).isNull() | (F.trim(F.col(col)) == "") | (F.lower(F.col(col)) == "null")
        ).count()
        pct = (null_ct / total_rows * 100) if total_rows > 0 else 0
        print(f"  {col:<30} null/empty: {null_ct:>4} ({pct:5.1f}%)")

# COMMAND ----------
# MAGIC %md ## 9. MLflow Quality Logging

# COMMAND ----------

try:
    import mlflow
    mlflow.set_experiment("/Users/dasdeepayan08@gmail.com/virtue-foundation-idp-hackathon")
    with mlflow.start_run(run_name="01_bronze_ingestion"):
        mlflow.log_param("source_file",     cfg.SOURCE_FILE_NAME)
        mlflow.log_param("dataset_version", cfg.DATASET_VERSION)
        mlflow.log_param("bronze_table",    cfg.BRONZE_TABLE)
        mlflow.log_param("csv_path",        CSV_PATH)
        mlflow.log_metric("total_rows",     total_rows)
        mlflow.log_metric("total_columns",  len(bronze_df.columns))
    print("✅  MLflow logged")
except Exception as e:
    print(f"MLflow skipped: {e}")

# COMMAND ----------
# MAGIC %md ## 10. Write Bronze Delta Table

# COMMAND ----------

(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("mergeSchema", "false")
    .option("delta.autoOptimize.optimizeWrite", "true")
    .option("delta.autoOptimize.autoCompact",   "true")
    .saveAsTable(cfg.BRONZE_TABLE)
)

readback = spark.table(cfg.BRONZE_TABLE).count()
assert readback == total_rows, f"Row count mismatch: wrote {total_rows}, read {readback}"
print(f"\n✅  Bronze table written: {cfg.BRONZE_TABLE}")
print(f"    Rows    : {readback:,}")
print(f"    Columns : {len(bronze_df.columns)}")

# COMMAND ----------
# MAGIC %md ## 11. Register Table Comment

# COMMAND ----------

spark.sql(f"""
    COMMENT ON TABLE {cfg.BRONZE_TABLE}
    IS 'Raw ingestion from Virtue Foundation Ghana v0.3 dataset — {total_rows} rows, 46 columns'
""")
print("✅  Table comment registered")

# COMMAND ----------
# MAGIC %md ## 12. Quick Data Verification

# COMMAND ----------

spark.table(cfg.BRONZE_TABLE).select(
    "name", "facilitytypeid", "address_city", "specialties", "capability"
).show(5, truncate=80)

# COMMAND ----------
# MAGIC %md ## 13. Sample Display

# COMMAND ----------

display(
    spark.table(cfg.BRONZE_TABLE)
    .select("unique_id", "name", "organization_type", "facilitytypeid",
            "address_city", "address_stateorregion",
            "specialties", "capability", "procedure", "equipment")
    .limit(10)
)

In [0]:
%sql
SELECT * FROM virtue_foundation.ghana.bronze_facilities_raw LIMIT 250

In [0]:
%sql
DESCRIBE TABLE EXTENDED virtue_foundation.ghana.bronze_facilities_raw

In [0]:
%sql
SHOW CREATE TABLE virtue_foundation.ghana.bronze_facilities_raw